## التقييم النهائي لأداء نظام استرجاع المعلومات

هذا الـ Notebook مخصص لإجراء التقييم النهائي والأساسي على مجموعة البيانات الكاملة، باستخدام معاملات BM25 المثبتة مسبقاً.

**المهام:**
1.  تحميل مجموعة البيانات والاستعلامات وأحكام الصلة (qrels).
2.  إجراء تقييم شامل (MAP, MRR, P@10, R@10) لكل نموذج من النماذج الأربعة الأساسية.
3.  عرض النتائج النهائية في جدول واحد وواضح.

### الخطوة 1: الإعدادات العامة وتحميل المكتبات

In [4]:
import requests
import ir_datasets
import pandas as pd
import numpy as np
from tqdm import tqdm
import sys
import os
import matplotlib.pyplot as plt
import seaborn as sns

# --- الإعدادات التي يمكنك تغييرها ---
DATASET_TO_EVALUATE = 'wikir/en1k/test' # غيّر هذه القيمة إلى 'antique' أو 'wikir/en1k'
QUERY_SAMPLE_SIZE = None # <<<--- تم التعيين إلى None لتشغيل التقييم على جميع الاستعلامات
# -----------------------------------

project_root = os.path.dirname(os.path.abspath(os.getcwd()))
sys.path.append(project_root)
from config import API_PORTS

# تحديد اسم مجموعة الاختبار الصحيح تلقائياً
if '/' in DATASET_TO_EVALUATE:
    IR_DATASET_NAME_TEST = f'{DATASET_TO_EVALUATE}'
else:
    IR_DATASET_NAME_TEST = f'{DATASET_TO_EVALUATE}'

SEARCH_API_URL = f"http://127.0.0.1:{API_PORTS['SEARCH']}/search/"
MODELS_TO_EVALUATE = ['tfidf', 'bm25', 'bert', 'hybrid']
BEST_K1 = 1.6 # قيمة k1 المثبتة
BEST_B = 0.75  # قيمة b المثبتة

### الخطوة 2: تحميل البيانات وتعريف دوال التقييم

In [5]:
print(f"Loading dataset: {IR_DATASET_NAME_TEST}...")
try:
    dataset = ir_datasets.load(IR_DATASET_NAME_TEST)
    queries = {q.query_id: q.text for q in dataset.queries_iter()}
    qrels = {}
    for qrel in dataset.qrels_iter():
        if qrel.relevance > 0:
            if qrel.query_id not in qrels:
                qrels[qrel.query_id] = set()
            qrels[qrel.query_id].add(qrel.doc_id)
except Exception as e:
    print(f"Could not load dataset {IR_DATASET_NAME_TEST}. Error: {e}")

query_ids_with_rels = [qid for qid in queries.keys() if qid in qrels]
query_ids_to_run = query_ids_with_rels[:QUERY_SAMPLE_SIZE] if QUERY_SAMPLE_SIZE else query_ids_with_rels
print(f"Loaded data. Running evaluation on {len(query_ids_to_run)} queries that have relevance judgments.")

def calculate_metrics(retrieved, relevant):
    if not retrieved or not relevant: return {'AP': 0.0, 'RR': 0.0, 'P@10': 0.0, 'R@10': 0.0}
    hits, sum_p, rr = 0, 0.0, 0.0; first = True
    for i, doc_id in enumerate(retrieved):
        if doc_id in relevant:
            if first: rr = 1 / (i + 1); first = False
            hits += 1; sum_p += hits / (i + 1)
    ap = sum_p / len(relevant)
    hits_at_10 = len(set(retrieved[:10]).intersection(relevant))
    p10 = hits_at_10 / 10.0
    r10 = hits_at_10 / len(relevant)
    return {'AP': ap, 'RR': rr, 'P@10': p10, 'R@10': r10}

def evaluate_model(queries_list, model_type):
    all_metrics = []
    pbar = tqdm(queries_list, desc=f"Evaluating {model_type}")
    for query_id in pbar:
        query_text, relevant_docs = queries.get(query_id), qrels.get(query_id, set())
        if not query_text or not relevant_docs: continue
        payload = {
            "dataset_name": DATASET_TO_EVALUATE, "query": query_text, "model_type": model_type,
            "top_k": 100, "k1": BEST_K1, "b": BEST_B, "enable_ner_reranking": False, "hybrid_bm25_weight": 0.8
        }
        try:
            r = requests.post(SEARCH_API_URL, json=payload, timeout=60)
            r.raise_for_status()
            retrieved_docs = [res['doc_id'] for res in r.json().get('results', [])]
            all_metrics.append(calculate_metrics(retrieved_docs, relevant_docs))
        except requests.exceptions.RequestException:
            all_metrics.append(calculate_metrics([], relevant_docs))
    return pd.DataFrame(all_metrics).mean().to_dict() if all_metrics else {}

Loading dataset: wikir/en1k/test...
Loaded data. Running evaluation on 100 queries that have relevance judgments.


### الخطوة 3: تنفيذ التقييم وعرض النتائج

In [6]:
print("--- Running Final Evaluation on All Queries ---")
final_results_list = []

for model_type in MODELS_TO_EVALUATE:
    metrics = evaluate_model(query_ids_to_run, model_type)
    metrics['Model'] = model_type
    final_results_list.append(metrics)

final_df = pd.DataFrame(final_results_list).rename(columns={'AP': 'MAP', 'RR': 'MRR'})

# عرض النتائج في جدول واحد نهائي
results_df = final_df.set_index('Model')[['MAP', 'MRR', 'P@10', 'R@10']]

print(f"\n--- Final Evaluation Results for '{DATASET_TO_EVALUATE}' ---")
display(results_df.style.format('{:.4f}')\
                      .background_gradient(cmap='viridis', axis=0)\
                      .set_caption("مقارنة أداء النماذج الأساسية"))

--- Running Final Evaluation on All Queries ---


Evaluating hybrid: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:37<00:00,  2.70it/s]


--- Final Evaluation Results for 'wikir/en1k/test' ---


,MAP,MRR,P@10,R@10
Model,,,,
tfidf,0.1609,0.6360,0.1950,0.1865
bm25,0.1703,0.6343,0.2050,0.1924
bert,0.1350,0.7415,0.1850,0.1576
hybrid,0.1838,0.7266,0.2160,0.2009
